In [5]:
# -----------------------------
# 03_train_efficientnet.ipynb
# -----------------------------

# Importations
import numpy as np
import torch
import os
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader, random_split
from torchvision import models, transforms
import torch.optim as optim
import mlflow
import mlflow.pytorch
from torchvision.models import efficientnet_b0, EfficientNet_B0_Weights


AttributeError: partially initialized module 'mlflow' has no attribute 'version' (most likely due to a circular import)

In [6]:
# -----------------------------
# Paramètres
# -----------------------------
# -----------------------------
# Paramètres
# -----------------------------
BATCH_SIZE = 16
EPOCHS = 15
LEARNING_RATE = 1e-3
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Résolution robuste du dossier processed (remonte l'arborescence depuis le cwd)
def find_processed_dir():
    cur = os.getcwd()
    while True:
        candidate = os.path.join(cur, "data", "processed")
        if os.path.isdir(candidate):
            return os.path.abspath(candidate)
        parent = os.path.dirname(cur)
        if parent == cur:
            break
        cur = parent
    # fallback: try relative to this notebook file location if possible
    # (Jupyter notebooks may start with different working dirs)
    possible = os.path.abspath(os.path.join('..', '..', 'data', 'processed'))
    if os.path.isdir(possible):
        return possible
    raise FileNotFoundError(f'Could not find data/processed starting from cwd={os.getcwd()}')

PROCESSED_DIR = find_processed_dir()
NUM_CLASSES = 2
IMAGE_SIZE = 224

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
MODEL_PATH = "efficientnet_model.pth"

IMAGE_SIZE = 224

In [7]:

# -----------------------------
# Chargement des données
# -----------------------------

# Si grayscale → 1 canal
# Transforme en tenseur PyTorch et ajoute dimension canal


# Charger les fichiers numpy
images = np.load(os.path.join(PROCESSED_DIR, "images.npy"))  # shape = [N,224,224,1]
labels = np.load(os.path.join(PROCESSED_DIR, "labels.npy"))
X = torch.tensor(images, dtype=torch.float32).unsqueeze(1)  # shape = [N,1,224,224]
y = torch.tensor(labels, dtype=torch.long)

# Créer Dataset et DataLoader
dataset = TensorDataset(X, y)
train_size = int(0.8 * len(dataset))
val_size = len(dataset) - train_size
train_dataset, val_dataset = random_split(dataset, [train_size, val_size])

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)

print("Train batch shape:", next(iter(train_loader))[0].shape)
print("Validation batch shape:", next(iter(val_loader))[0].shape)



Train batch shape: torch.Size([16, 1, 224, 224, 1])
Validation batch shape: torch.Size([16, 1, 224, 224, 1])


In [8]:
# -----------------------------
# Définition du modèle EfficientNet (B0)
# -----------------------------
#on prend les poids pré-entraînés sur ImageNet 
weights = EfficientNet_B0_Weights.DEFAULT
#crée le modèle EfficientNet-B0 et charge ces poids.
model = efficientnet_b0(weights=weights)

# Adapter la première couche pour 1 canal
old_conv = model.features[0][0]
new_conv = nn.Conv2d(
    in_channels=1,
    out_channels=old_conv.out_channels,
    kernel_size=old_conv.kernel_size,
    stride=old_conv.stride,
    padding=old_conv.padding,
    #BN fait le rôle de biais automatiquement (Le biais permet de décaler la sortie, même si toutes les entrées sont nulles.)
    bias=False
)
with torch.no_grad():
    #Les poids de la nouvelle couche sont initialisés à la moyenne des 3 canaux originaux.
    new_conv.weight[:, 0, :, :] = old_conv.weight.mean(dim=1)
    #On remplace la première couche originale par cette nouvelle couche adaptée au grayscale
model.features[0][0] = new_conv

# Modifier la dernière couche pour 2 classes ,2 couches dans le Sequential
model.classifier[1] = nn.Linear(model.classifier[1].in_features, 2)
model = model.to(DEVICE)

In [9]:

# Loss et optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)

In [10]:
# -----------------------------
# Configuration MLflow
# -----------------------------
mlflow.set_tracking_uri("file:./mlruns")
mlflow.set_experiment("EfficientNet_Model")


2026/02/02 22:01:21 INFO mlflow.tracking.fluent: Experiment with name 'EfficientNet_Model' does not exist. Creating a new experiment.


<Experiment: artifact_location='file:///c:/Users/USER/Desktop/ecg-classification/src/train/mlruns/390530276022161732', creation_time=1770066081637, experiment_id='390530276022161732', last_update_time=1770066081637, lifecycle_stage='active', name='EfficientNet_Model', tags={}>

In [11]:
# -----------------------------
# Réinitialisation des runs fantômes
# -----------------------------
# Vérifie si un run fantôme existe

if mlflow.active_run() is not None:
    print("Run actif trouvé :", mlflow.active_run().info.run_id)
    mlflow.end_run()
    print("Run actif terminé ✅")

In [12]:
# -----------------------------
# Entraînement avec MLflow
# -----------------------------
with mlflow.start_run(run_name="EfficientNet_Run") as run:

    # Log des paramètres
    mlflow.log_param("model", "EfficientNet_B0")
    mlflow.log_param("epochs", EPOCHS)
    mlflow.log_param("device", str(DEVICE))

    for epoch in range(EPOCHS):
        train_loss = 0.0
        train_acc = 0.0
        val_loss = 0.0
        val_acc = 0.0
        
        # ======= Boucle d'entraînement =======
        model.train()
        for inputs, targets in train_loader:
            inputs, targets = inputs.to(DEVICE), targets.to(DEVICE)
            
            # Supprimer la dimension en trop si nécessaire
            if inputs.dim() == 5 and inputs.size(-1) == 1:
                inputs = inputs.squeeze(-1)
            
            optimizer.zero_grad()
            outputs = model(inputs)
            #  Calcul de la perte
            loss = criterion(outputs, targets)
            #Backpropagation calcule les gradients
            loss.backward()
            #Mise à jour des poids
            optimizer.step()
           #On cumule loss et accuracy pour tout le batch.
            train_loss += loss.item() * inputs.size(0)
            _, predicted = torch.max(outputs, 1)
            total = targets.size(0)
            train_acc += (predicted == targets).sum().item()
        #Moyenne sur tout le dataset → métriques finales de l’epoch.
        train_loss /= len(train_loader.dataset)
        train_acc /= len(train_loader.dataset)

        # ======= Boucle de validation =======
        model.eval()
        # pas de calcul de gradients → économise mémoire et accélère
        with torch.no_grad():
            for inputs, targets in val_loader:
                inputs, targets = inputs.to(DEVICE), targets.to(DEVICE)
                if inputs.dim() == 5 and inputs.size(-1) == 1:
                    inputs = inputs.squeeze(-1)
                outputs = model(inputs)
                loss = criterion(outputs, targets)
                val_loss += loss.item() * inputs.size(0)
                _, predicted = torch.max(outputs, 1)
                val_acc += (predicted == targets).sum().item()

        val_loss /= len(val_loader.dataset)
        val_acc /= len(val_loader.dataset)
        print(f"Epoch [{epoch+1}/{EPOCHS}] Train Loss: {train_loss:.4f} Train Acc: {train_acc:.4f} "
              f"Val Loss: {val_loss:.4f} Val Acc: {val_acc:.4f}")
        # Log metrics dans MLflow
        mlflow.log_metric("train_loss", train_loss, step=epoch)
        mlflow.log_metric("train_acc", train_acc, step=epoch)
        mlflow.log_metric("val_loss", val_loss, step=epoch)
        mlflow.log_metric("val_acc", val_acc, step=epoch)

    # ======= Sauvegarde du modèle =======
    model.eval()
    mlflow.pytorch.log_model(
        pytorch_model=model,
        artifact_path="efficientnet_model",
    )

    print("Run MLflow terminé avec succès :", run.info.run_id)

Epoch [1/15] Train Loss: 0.2507 Train Acc: 0.8935 Val Loss: 3.3349 Val Acc: 0.4423
Epoch [2/15] Train Loss: 0.1497 Train Acc: 0.9346 Val Loss: 0.1149 Val Acc: 0.9808
Epoch [3/15] Train Loss: 0.0897 Train Acc: 0.9734 Val Loss: 0.0793 Val Acc: 1.0000
Epoch [4/15] Train Loss: 0.0867 Train Acc: 0.9661 Val Loss: 0.0183 Val Acc: 1.0000
Epoch [5/15] Train Loss: 0.0563 Train Acc: 0.9879 Val Loss: 0.0146 Val Acc: 1.0000
Epoch [6/15] Train Loss: 0.1236 Train Acc: 0.9516 Val Loss: 0.1415 Val Acc: 0.9135
Epoch [7/15] Train Loss: 0.1274 Train Acc: 0.9588 Val Loss: 0.0370 Val Acc: 0.9904
Epoch [8/15] Train Loss: 0.0562 Train Acc: 0.9758 Val Loss: 0.1869 Val Acc: 0.9135
Epoch [9/15] Train Loss: 0.0255 Train Acc: 0.9952 Val Loss: 0.0410 Val Acc: 0.9904
Epoch [10/15] Train Loss: 0.0348 Train Acc: 0.9927 Val Loss: 0.0298 Val Acc: 0.9904
Epoch [11/15] Train Loss: 0.0423 Train Acc: 0.9903 Val Loss: 0.0103 Val Acc: 1.0000
Epoch [12/15] Train Loss: 0.0101 Train Acc: 0.9976 Val Loss: 0.0299 Val Acc: 0.9808
E